### Dependencias y Carga

In [1]:
import pandas as pd
import matplotlib.colors as mcolors
import altair as alt

df = pd.read_csv('songs.csv', nrows=499000, encoding='latin1') 
df = df.dropna(subset=['year', 'valence', 'energy'])
df = df[(df['year']>=1970) & (df['year']<=2023)]


### Procesamiento

In [2]:
def clasificar_emocion(row):
    v = row['valence']
    e = row['energy']
    if v >= 0.5 and e >= 0.5: return 'Euforia'
    if v < 0.5 and e >= 0.5: return 'Ansiedad/Estrés'
    if v < 0.5 and e < 0.5: return 'Melancolía'
    return 'Calma'

df['Emocion'] = df.apply(clasificar_emocion, axis=1)

# agrupar por año y calcular porcentajes
counts = df.groupby(['year', 'Emocion']).size().reset_index(name='cantidad')
counts['porcentaje'] = counts.groupby('year')['cantidad'].transform(lambda x: x / x.sum())

### Gráfico

In [3]:
# streamgraph
stream = alt.Chart(counts).mark_area().encode(
    x=alt.X('year:O', title='Año'),
    y=alt.Y('cantidad:Q', 
        stack='normalize', 
        axis=alt.Axis(format='%', values=[0, 0.25, 0.50, 0.75, 1.0], title=None)),
    color=alt.Color('Emocion:N', scale=alt.Scale(scheme='category10'), title='Estado Emocional'),
    tooltip=['year', 'Emocion', 'porcentaje']
).properties(
    width=800, 
    height=400, 
    title={
        "text": ["Evolución del Perfil Psicológico Musical (1970-2023)"],
        "subtitle": ["Distribución porcentual de emociones basado en el modelo de Russell."],
        "color": "black",
        "subtitleColor": "gray"
    }
)

stream = stream.configure(
    background='transparent'
)

stream

alt.Chart(...)